### FEMA -- Data Preprocessing

- Fix data quality issues and prepare clean datasets for feature engineering & modelling

### Setup & import libraries


In [ ]:

import os, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

## Define raw and processed paths

DATA_RAW = os.path.join("..", "data", "raw")
DATA_PROCESSED = os.path.join("..", "data", "processed")

os.makedirs(DATA_PROCESSED, exist_ok=True)

## Load the datasets

declarations = pd.read_csv(os.path.join(DATA_RAW, "declarations.csv"))
public_assistance = pd.read_csv(os.path.join(DATA_RAW, "public_assistance.csv"))
disaster_summaries = pd.read_csv(os.path.join(DATA_RAW, "disaster_summaries.csv"))

### Step 1 - Cleaning Exercise: Declarations Dataset

In [33]:
# overview
declarations.head()

,disasterNumber,state,declarationType,incidentType,declarationDate,incidentBeginDate,incidentEndDate,fyDeclared,designatedArea,declarationRequestNumber
0,3610,PR,EM,Severe Storm,2024-08-13T00:00:00.000Z,2024-08-13T00:00:00.000Z,2024-08-16T00:00:00.000Z,2024,Adjuntas (Municipio),24124
1,5529,OR,FM,Fire,2024-08-09T00:00:00.000Z,2024-08-08T00:00:00.000Z,NaN,2024,Washington (County),24122
2,5528,OR,FM,Fire,2024-08-06T00:00:00.000Z,2024-08-04T00:00:00.000Z,NaN,2024,Jefferson (County),24116
3,5527,OR,FM,Fire,2024-08-02T00:00:00.000Z,2024-08-02T00:00:00.000Z,NaN,2024,Deschutes (County),24111
4,3610,PR,EM,Severe Storm,2024-08-13T00:00:00.000Z,2024-08-13T00:00:00.000Z,2024-08-16T00:00:00.000Z,2024,Aguada (Municipio),24124


In [34]:
declarations_clean = declarations.copy()

# STEP 1A - Standardize column names
declarations_clean.columns = (
    declarations_clean.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.lower()
)

In [35]:
# STEP 1B - Convert date columns to datetime


date_cols = [
    "declarationdate",
    "incidentbegindate",
    "incidentenddate"
    
]

for col in date_cols:
    if col in declarations_clean.columns:
        declarations_clean[col] = pd.to_datetime(
            declarations_clean[col],
            errors="coerce"
        )

In [36]:
# STEP 1E - Clean categorical columns

cat_cols = [
    "state",
    "declarationtype",
    "incidenttype",
    "designatedarea",
    "region"
]

for col in cat_cols:
    if col in declarations_clean.columns:
        declarations_clean[col] = (
            declarations_clean[col]
            .astype(str)
            .str.strip()
            .str.title()
        )

In [37]:

# STEP 1F - Remove duplicate disaster records if any


declarations_clean = declarations_clean.drop_duplicates()

print(f"Declarations columns data-types:\n{declarations_clean.dtypes}")
print("\nMissing values:")
print(declarations_clean.isnull().sum())

declarations_clean.head()

Declarations columns data-types:
disasternumber                            int64
state                                       str
declarationtype                             str
incidenttype                                str
declarationdate             datetime64[us, UTC]
incidentbegindate           datetime64[us, UTC]
incidentenddate             datetime64[us, UTC]
fydeclared                                int64
designatedarea                              str
declarationrequestnumber                  int64
dtype: object

Missing values:
disasternumber                0
state                         0
declarationtype               0
incidenttype                  0
declarationdate               0
incidentbegindate             0
incidentenddate             534
fydeclared                    0
designatedarea                0
declarationrequestnumber      0
dtype: int64


,disasternumber,state,declarationtype,incidenttype,declarationdate,incidentbegindate,incidentenddate,fydeclared,designatedarea,declarationrequestnumber
0,3610,Pr,Em,Severe Storm,2024-08-13 00:00:00+00:00,2024-08-13 00:00:00+00:00,2024-08-16 00:00:00+00:00,2024,Adjuntas (Municipio),24124
1,5529,Or,Fm,Fire,2024-08-09 00:00:00+00:00,2024-08-08 00:00:00+00:00,NaT,2024,Washington (County),24122
2,5528,Or,Fm,Fire,2024-08-06 00:00:00+00:00,2024-08-04 00:00:00+00:00,NaT,2024,Jefferson (County),24116
3,5527,Or,Fm,Fire,2024-08-02 00:00:00+00:00,2024-08-02 00:00:00+00:00,NaT,2024,Deschutes (County),24111
4,3610,Pr,Em,Severe Storm,2024-08-13 00:00:00+00:00,2024-08-13 00:00:00+00:00,2024-08-16 00:00:00+00:00,2024,Aguada (Municipio),24124


### Step 2 - Cleaning Exercise: Public Assistance Dataset

In [38]:
print(f"Public Assistance columns data-types:\n{public_assistance.dtypes}") 
print(public_assistance.isnull().sum())
public_assistance.head()


Public Assistance columns data-types:
disasterNumber             int64
stateAbbreviation            str
projectAmount            float64
federalShareObligated    float64
totalObligated           float64
projectSize                  str
damageCategoryCode           str
applicationTitle             str
dtype: object
disasterNumber           0
stateAbbreviation        0
projectAmount            0
federalShareObligated    0
totalObligated           0
projectSize              0
damageCategoryCode       0
applicationTitle         0
dtype: int64


,disasterNumber,stateAbbreviation,projectAmount,federalShareObligated,totalObligated,projectSize,damageCategoryCode,applicationTitle
0,1361,WA,1203.00,902.25,957.10,Small,B,(PW# 81) INSPECTION OF TOWN BUILDINGS & UTILITIES
1,1603,LA,15156787.07,15156787.07,15312129.35,Large,E,(PW# 18773) CONTENTS ROLLUP-MULTIPLE SITES; MU...
2,3582,MS,159127.24,119345.43,119345.43,Small,B,MS- State Department of Health EPM-Period 1 - ...
3,1817,WA,6847.49,5135.62,5135.62,Small,C,(PW# 537) SEBC005 - S. Emery Ave. Road/Shoulde...
4,3582,MS,118703.32,89027.49,89027.49,Small,B,MS- State Department of Health EPM-Period 3 - ...


In [39]:
# save a copy

pa_clean = public_assistance.copy()

# STEP 2A - Standardize column names
pa_clean.columns = (
    pa_clean.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.lower()
)

In [40]:
# STEP 2C - Clean disaster number

if "disasternumber" in pa_clean.columns:
    pa_clean["disasternumber"] = pa_clean["disasternumber"].astype(int)

In [41]:
# STEP 2D - Clean funding columns

funding_cols = [
    "projectamount",
    "federalshareobligated",
    "totalobligated"
]

for col in funding_cols:
    if col in pa_clean.columns:
        pa_clean[col] = pd.to_numeric(pa_clean[col], errors="coerce")
        pa_clean[col] = pa_clean[col].fillna(0)
        pa_clean[col] = pa_clean[col].clip(lower=0)

In [42]:
# STEP 2E - Clean categorical columns

pa_cat_cols = [

"projectSize"                  
"damageCategoryCode"          
"applicationTitle"
"stateAbbreviation"
    
]

for col in pa_cat_cols:
    if col in pa_clean.columns:
        pa_clean[col] = (
            pa_clean[col]
            .astype(str)
            .str.strip()
            .str.title()
        )

In [43]:

# STEP 2F - Remove duplicate rows

pa_clean = pa_clean.drop_duplicates()
print(f"Public Assistance columns data-types:\n{pa_clean.dtypes}")
print("\nMissing values:")
print(pa_clean.isnull().sum())

pa_clean.head()


Public Assistance columns data-types:
disasternumber             int64
stateabbreviation            str
projectamount            float64
federalshareobligated    float64
totalobligated           float64
projectsize                  str
damagecategorycode           str
applicationtitle             str
dtype: object

Missing values:
disasternumber           0
stateabbreviation        0
projectamount            0
federalshareobligated    0
totalobligated           0
projectsize              0
damagecategorycode       0
applicationtitle         0
dtype: int64


,disasternumber,stateabbreviation,projectamount,federalshareobligated,totalobligated,projectsize,damagecategorycode,applicationtitle
0,1361,WA,1203.00,902.25,957.10,Small,B,(PW# 81) INSPECTION OF TOWN BUILDINGS & UTILITIES
1,1603,LA,15156787.07,15156787.07,15312129.35,Large,E,(PW# 18773) CONTENTS ROLLUP-MULTIPLE SITES; MU...
2,3582,MS,159127.24,119345.43,119345.43,Small,B,MS- State Department of Health EPM-Period 1 - ...
3,1817,WA,6847.49,5135.62,5135.62,Small,C,(PW# 537) SEBC005 - S. Emery Ave. Road/Shoulde...
4,3582,MS,118703.32,89027.49,89027.49,Small,B,MS- State Department of Health EPM-Period 3 - ...



### Step 3 - Cleaning Exercise: Disaster Summaries Dataset

In [44]:
# overview
print(f"Disaster Summaries shape: {disaster_summaries.shape}")
print(f"Disaster Summaries columns data-types:\n{disaster_summaries.dtypes}")
print("\nMissing values:")
print(disaster_summaries.isnull().sum())
disaster_summaries.head()

Disaster Summaries shape: (3945, 11)
Disaster Summaries columns data-types:
disasterNumber                  int64
totalNumberIaApproved         float64
totalAmountIhpApproved        float64
totalAmountHaApproved         float64
totalAmountOnaApproved        float64
totalObligatedAmountPa        float64
totalObligatedAmountCatAb     float64
totalObligatedAmountCatC2g    float64
totalObligatedAmountHmgp      float64
paLoadDate                        str
iaLoadDate                        str
dtype: object

Missing values:
disasterNumber                   0
totalNumberIaApproved         3339
totalAmountIhpApproved        3339
totalAmountHaApproved         3395
totalAmountOnaApproved        3341
totalObligatedAmountPa         956
totalObligatedAmountCatAb     1217
totalObligatedAmountCatC2g    2389
totalObligatedAmountHmgp      1102
paLoadDate                     956
iaLoadDate                    3339
dtype: int64


,disasterNumber,totalNumberIaApproved,totalAmountIhpApproved,totalAmountHaApproved,totalAmountOnaApproved,totalObligatedAmountPa,totalObligatedAmountCatAb,totalObligatedAmountCatC2g,totalObligatedAmountHmgp,paLoadDate,iaLoadDate
0,3601,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3602,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1802,NaN,NaN,NaN,NaN,1.881275e+07,1.342309e+07,4.869890e+06,2710679.0,2026-02-06T00:00:00.000Z,NaN
3,1292,NaN,NaN,NaN,NaN,2.981058e+08,1.436027e+08,1.504852e+08,67665966.0,2026-02-06T00:00:00.000Z,NaN
4,3163,NaN,NaN,NaN,NaN,5.380816e+06,5.369404e+06,NaN,NaN,2026-02-06T00:00:00.000Z,NaN


In [45]:
# make a copy
summaries_clean = disaster_summaries.copy()

# STEP 3A - Standardize column names
summaries_clean.columns = (
    summaries_clean.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.lower()
)

In [46]:
# STEP 3B - Convert date columns

summary_date_cols = [
    "paLoadDate",                   
    "iaLoadDate"
]

for col in summary_date_cols:
    if col in summaries_clean.columns:
        summaries_clean[col] = pd.to_datetime(summaries_clean[col], errors="coerce")
        

In [47]:
# STEP 3C - Clean disaster number

if "disasternumber" in summaries_clean.columns:
    summaries_clean["disasternumber"] = summaries_clean["disasternumber"].astype(int)


In [48]:

# STEP 3D - Clean cost / obligation columns

summary_money_cols = [
    
"totalNumberIaApproved",        
"totalAmountIhpApproved",        
"totalAmountHaApproved",         
"totalAmountOnaApproved",        
"totalObligatedAmountPa",         
"totalObligatedAmountCatAb"     
"totalObligatedAmountCatC2g",    
"totalObligatedAmountHmg"
]

for col in summary_money_cols:
    if col in summaries_clean.columns:
        summaries_clean[col] = pd.to_numeric(summaries_clean[col], errors="coerce")

In [49]:
# STEP 3E - Fill numeric missing values with 0 where suitable

for col in summaries_clean.select_dtypes(include=["int64", "float64"]).columns:
    summaries_clean[col] = summaries_clean[col].fillna(0)

In [50]:
# STEP 3G - Remove duplicate rows

summaries_clean = summaries_clean.drop_duplicates()
print(f"Disaster Summaries columns data-types:\n{summaries_clean.dtypes}")
print("\nMissing values:")
print(summaries_clean.isnull().sum())

summaries_clean.head()


Disaster Summaries columns data-types:
disasternumber                  int64
totalnumberiaapproved         float64
totalamountihpapproved        float64
totalamounthaapproved         float64
totalamountonaapproved        float64
totalobligatedamountpa        float64
totalobligatedamountcatab     float64
totalobligatedamountcatc2g    float64
totalobligatedamounthmgp      float64
paloaddate                        str
ialoaddate                        str
dtype: object

Missing values:
disasternumber                   0
totalnumberiaapproved            0
totalamountihpapproved           0
totalamounthaapproved            0
totalamountonaapproved           0
totalobligatedamountpa           0
totalobligatedamountcatab        0
totalobligatedamountcatc2g       0
totalobligatedamounthmgp         0
paloaddate                     956
ialoaddate                    3339
dtype: int64


,disasternumber,totalnumberiaapproved,totalamountihpapproved,totalamounthaapproved,totalamountonaapproved,totalobligatedamountpa,totalobligatedamountcatab,totalobligatedamountcatc2g,totalobligatedamounthmgp,paloaddate,ialoaddate
0,3601,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,NaN,NaN
1,3602,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,NaN,NaN
2,1802,0.0,0.0,0.0,0.0,1.881275e+07,1.342309e+07,4.869890e+06,2710679.0,2026-02-06T00:00:00.000Z,NaN
3,1292,0.0,0.0,0.0,0.0,2.981058e+08,1.436027e+08,1.504852e+08,67665966.0,2026-02-06T00:00:00.000Z,NaN
4,3163,0.0,0.0,0.0,0.0,5.380816e+06,5.369404e+06,0.000000e+00,0.0,2026-02-06T00:00:00.000Z,NaN


In [51]:
### Step 4 - Referential Integrity

In [52]:
valid_disasters = set(declarations_clean["disasternumber"])

before_pa = len(pa_clean)
before_summaries = len(summaries_clean)

pa_clean = pa_clean[pa_clean["disasternumber"].isin(valid_disasters)]
summaries_clean = summaries_clean[summaries_clean["disasternumber"].isin(valid_disasters)]

print(f"Public Assistance rows dropped: {before_pa - len(pa_clean)}")
print(f"Disaster Summary rows dropped: {before_summaries - len(summaries_clean)}")
print("Referential integrity check completed")

Public Assistance rows dropped: 0
Disaster Summary rows dropped: 0
Referential integrity check completed


In [53]:
###  Step 5 - Validate Row Counts

In [54]:
print("Row count - raw vs clean")

for name, raw, clean in [
    ("declarations", declarations, declarations_clean),
    ("public_assistance", public_assistance, pa_clean),
    ("disaster_summaries", disaster_summaries, summaries_clean)
]:
    print(f"{name:<22}: {len(raw):>7,} -> {len(clean):>7,}")

Row count - raw vs clean
declarations          :  69,905 ->  69,877
public_assistance     : 812,867 -> 812,858
disaster_summaries    :   3,945 ->   3,945


### Step 6 - Save Clean Datasets

In [55]:
files = {
    "declarations_clean.csv": declarations_clean,
    "public_assistance_clean.csv": pa_clean,
    "disaster_summaries_clean.csv": summaries_clean
}

for fname, df in files.items():
    df.to_csv(os.path.join(DATA_PROCESSED, fname), index=False)
    print(f"{fname} saved successfully")

declarations_clean.csv saved successfully
public_assistance_clean.csv saved successfully
disaster_summaries_clean.csv saved successfully
